# WNBA Prop Predictor

Compares your model's predicted probability of a prop hitting against the
sportsbook's implied probability from the odds you enter, then tracks
closing-line value (CLV) and actual outcomes over time to recalibrate
itself.

**How to use this notebook:**
1. Run the cells top to bottom (Runtime > Run all works fine).
2. In the form that appears, pick the player's team / opponent team / prop
   type from the dropdowns, type in the player's name, the line, the odds,
   the game total and spread.
3. Click **Auto-Fetch Data** -- it tries to pull season/L20/L10/L5 stats,
   minutes, usage, and opponent pace/defense ranks from public WNBA data
   sources. Every field it fills in is still an editable box, so if a fetch
   fails (these are unofficial endpoints and can change), just type the
   number in yourself and keep going.
4. Click **Run Prediction** to see the model probability vs. the market's
   implied probability and the edge between them.
5. Click **Log This Prediction** to save it. After the game, come back,
   pick the bet from the pending list, enter the actual stat result and
   (optionally) the closing odds, and click **Record Outcome**.
6. Once you've logged ~20+ resolved bets, click **Recalibrate Model** to
   have it fit a probability calibrator and re-weight the recency windows
   based on what has actually been predictive.

**Data note:** the auto-fetch hits `stats.wnba.com` and ESPN's public
endpoints. They're unofficial (reverse-engineered) and can rate-limit or
change shape without notice -- that's exactly why every value is manually
editable. Nothing here is guaranteed uptime; treat auto-fetch as a
convenience, not a dependency.


## 1. Install dependencies

In [ ]:
!pip -q install ipywidgets pandas numpy scipy scikit-learn requests


## 2. Get the predictor code

Pulls the `wnba_predictor` package from GitHub. If you already have the repo
cloned locally (e.g. running outside Colab), this cell just leaves your copy
alone.


In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/ianbjorgum35-creator/WNBA.git"
BRANCH = "claude/wnba-update"  # update this if the branch above is merged/deleted
CLONE_DIR = "wnba_repo"

def _clone(branch=None):
    cmd = ["git", "clone"]
    if branch:
        cmd += ["--branch", branch, "--single-branch"]
    cmd += [REPO_URL, CLONE_DIR]
    return subprocess.run(cmd, capture_output=True, text=True)

if os.path.isdir(os.path.join(CLONE_DIR, "wnba_predictor")):
    print("Repo already present, pulling latest...")
    subprocess.run(["git", "-C", CLONE_DIR, "pull"], check=False)
else:
    result = _clone(BRANCH)
    print(result.stdout, result.stderr)
    if result.returncode != 0:
        print(f"Clone of branch '{BRANCH}' failed (likely merged/renamed/deleted since this "
              "notebook was written) -- falling back to the repo's default branch.")
        subprocess.run(["rm", "-rf", CLONE_DIR])
        result = _clone(None)
        print(result.stdout, result.stderr)

if CLONE_DIR not in sys.path:
    sys.path.insert(0, CLONE_DIR)

if not os.path.isdir(os.path.join(CLONE_DIR, "wnba_predictor")):
    raise RuntimeError(
        "Could not find wnba_predictor/ after cloning. If this repo is private, either make it "
        "public or upload the wnba_predictor/ folder yourself (Colab: folder icon on the left -> "
        "upload) into this notebook's working directory, then re-run this cell."
    )
print("wnba_predictor package found -- ready for the next cell.")


## 3. (Optional) Persist your bet log to Google Drive

Colab's local disk is wiped when the runtime recycles. Mount Drive and point
the bet log there so your CLV/outcome history -- and therefore the model's
learning -- survives across sessions. Skip this cell if you're fine with a
session-only log.


In [ ]:
USE_DRIVE = False  # flip to True to persist your bet log across sessions

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = "/content/drive/MyDrive/wnba_prop_predictor"
    os.makedirs(DATA_DIR, exist_ok=True)
else:
    DATA_DIR = "data"

BET_LOG_PATH = os.path.join(DATA_DIR, "bet_log.csv")
WEIGHTS_PATH = os.path.join(DATA_DIR, "learned_weights.json")
print("Bet log will be stored at:", BET_LOG_PATH)


## 4. Launch the predictor

In [ ]:
from wnba_predictor import ui

predictor = ui.launch(bet_log_path=BET_LOG_PATH, weights_path=WEIGHTS_PATH)


## 5. (Optional) Inspect your model's calibration directly

Run this any time to see your win rate, Brier score, log loss, average CLV,
and hit-rate-by-predicted-probability-bucket without going through the UI
button.


In [ ]:
from wnba_predictor import clv
import json

print(json.dumps(clv.calibration_report(BET_LOG_PATH), indent=2, default=str))
